# GNN GraphSAINT-RW (Multi-PNA-EU) — Colab 실행 노트북

**baseline과 달라진 점**
- 샘플링 방식: `LinkNeighborLoader` → `GraphSAINTRandomWalkSampler` (subgraph sampling)
- 새 하이퍼파라미터: `walk_length`, `num_steps`, `saint_batch_size`
- seed 엣지 개념 없음 → `AddEgoIds` 미사용, arange ID로 test 엣지 식별

**실행 전 체크리스트**
- [ ] 런타임 → GPU (T4 이상) 설정
- [ ] Google Drive에 `Graph_AML/data/processed/gnn_inputs/` 아래 04번 산출물 3개 존재 확인
- [ ] **2번 셀** 경로 설정 확인
- [ ] **5번 셀** 실험 설정 확인

## 0. GPU 확인

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 런타임 유형 변경 → GPU로 전환하세요.")

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정 ← 여기만 수정

In [ ]:
from pathlib import Path

# ============================================================
# 팀원 Drive 구조에 맞게 수정하세요
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/Graph_AML")
REPO_URL   = "https://github.com/JKYUPSYCHE/Graph_AML"
BRANCH     = "gnn/graphsaint"

EXPERIMENT = "GNN-01"
RUN        = "r01"
# ============================================================

GNN_INPUTS = DRIVE_BASE / "data" / "processed" / "gnn_inputs"
GNN_DIR    = DRIVE_BASE / "gnn"
RUN_DIR    = GNN_DIR / EXPERIMENT / RUN

for sub in ['logs', 'models', 'runs', 'feature_importance']:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

required = [
    GNN_INPUTS / "formatted_transactions_gf.csv",
    GNN_INPUTS / "formatted_transactions.csv",
    GNN_INPUTS / "account_mapping.csv",
]
all_ok = True
for f in required:
    exists = f.exists()
    print(f"{'[OK]' if exists else '[MISSING]'} {f.name}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("04_gnn_graph_process.ipynb 산출물이 없습니다.")

print("\n경로 설정 완료")
print("GNN_INPUTS :", GNN_INPUTS)
print("RUN_DIR    :", RUN_DIR)

## 3. 레포 클론 & PyG 설치

In [ ]:
import os

REPO_DIR = Path("/content/Graph_AML")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
else:
    os.system(f"git -C {REPO_DIR} fetch origin")
    os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
    os.system(f"git -C {REPO_DIR} pull origin {BRANCH}")

print("레포 준비 완료:", REPO_DIR)

In [ ]:
import torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"설치 대상: torch={torch_ver}, cuda={cuda_tag}")

os.system("pip install -q torch_geometric")
os.system(f"pip install -q pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html")
os.system("pip install -q psutil tqdm tensorboard")
print("패키지 설치 완료")

## 4. 모듈 경로 설정

In [ ]:
import sys

GRAPHSAINT_DIR = REPO_DIR / "gnn" / "graphsaint"
BASELINE_DIR   = REPO_DIR / "gnn" / "baseline"

os.chdir(GRAPHSAINT_DIR)

# insert 순서 역순으로 — 마지막에 insert(0)된 게 앞으로 옴
# 최종 순서: GRAPHSAINT_DIR → BASELINE_DIR → REPO_DIR
for p in [str(REPO_DIR), str(BASELINE_DIR), str(GRAPHSAINT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("CWD        :", os.getcwd())
print("sys.path[0]:", sys.path[0])
print("sys.path[1]:", sys.path[1])

## 5. 실험 설정 ← 여기만 수정

In [ ]:
from types import SimpleNamespace

# ============================================================
# 실험 설정 — 필요에 따라 수정
# ============================================================
args = SimpleNamespace(
    # 모델
    model      = 'pna',
    data       = 'Small_HI',
    seed       = 42,

    # 학습
    n_epochs   = 100,
    patience   = 20,

    # GraphSAINT 샘플링 파라미터
    walk_length      = 6,    # 랜덤워크 길이
    num_steps        = 100,  # 에폭당 배치 수 (전체 학습 엣지의 ~10% 커버)
    saint_batch_size = 1000, # 워크당 시작 노드 수 → 배치당 fraud ~10개 확보

    # Multi-PNA-EU 핵심 플래그
    emlps         = True,
    reverse_mp    = True,    # True: 양방향 MP (hetero 변환은 배치 내 on-the-fly)
    ports         = True,
    reverse_ports = True,    # False: in_port만 사용
    tds           = False,
    ego           = False,   # GraphSAINT는 seed 엣지 개념 없음

    # LR Scheduler (ReduceLROnPlateau)
    lr_factor   = 0.5,       # val F1 정체 시 LR 감소 비율 (new_lr = lr * factor)
    lr_patience = 5,         # LR 감소 전 정체 허용 에폭 수

    # 저장/추론
    save_model  = True,
    unique_name = 'small_hi_graphsaint_rw_wl6_ns100_bs1000_lrs',
    finetune    = False,
    inference   = False,

    tqdm = False,
)
# ============================================================

data_config = {
    "paths": {
        "aml_data"      : str(DRIVE_BASE / "data"),
        "gnn_inputs"    : str(GNN_INPUTS),
        "model_to_load" : str(RUN_DIR / "models"),
        "model_to_save" : str(RUN_DIR / "models"),
        "tb_log_dir"    : str(RUN_DIR / "runs"),
    }
}

print(f"모델      : {args.model}")
print(f"데이터    : {args.data}")
print(f"에폭      : {args.n_epochs}  |  patience: {args.patience}")
print(f"GraphSAINT: walk_length={args.walk_length}, num_steps={args.num_steps}, saint_batch_size={args.saint_batch_size}")
print(f"플래그    : emlps={args.emlps}, reverse_mp={args.reverse_mp}, ports={args.ports}, tds={args.tds}")
print(f"LR Sched  : factor={args.lr_factor}, patience={args.lr_patience}")
print(f"저장 경로 : {RUN_DIR}")

## 6. 데이터 로딩

In [ ]:
import time
import logging
from copy import copy
from utils import set_seed
from util import logger_setup
from data_loading import get_data

LOG_DIR = RUN_DIR / 'logs'
logger_setup(log_dir=LOG_DIR, log_name=args.unique_name)
set_seed(args.seed)

logging.info("데이터 로딩 시작")
t1 = time.perf_counter()

# GraphSAINT 샘플러는 homo Data만 받으므로 reverse_mp를 False로 해서 homo로 로딩
# hetero 변환은 학습 루프 내부에서 배치마다 on-the-fly로 수행
_args_homo = copy(args)
_args_homo.reverse_mp = False
tr_data, val_data, te_data, tr_inds, val_inds, te_inds = get_data(_args_homo, data_config)

t2 = time.perf_counter()
data_load_elapsed = t2 - t1
logging.info(f"데이터 로딩 완료: {data_load_elapsed:.2f}s")

## 7. 학습

> GraphSAINT는 랜덤워크 기반 서브그래프를 매 배치마다 샘플링합니다.  
> `num_steps`가 에폭당 배치 수, `saint_batch_size × walk_length`가 서브그래프 크기에 영향을 줍니다.

In [ ]:
import psutil
import torch
from training import train_gnn

_m, _s = divmod(int(data_load_elapsed), 60)
print(f"데이터 로딩 소요 시간: {_m:02d}분 {_s:02d}초")

cpu_pct = psutil.cpu_percent(interval=1)
ram     = psutil.virtual_memory()
print(f"CPU 사용률 : {cpu_pct:.1f}%")
print(f"RAM        : {ram.used / 1024**3:.1f} / {ram.total / 1024**3:.1f} GB  ({ram.percent:.1f}%)")

if torch.cuda.is_available():
    dev   = torch.cuda.current_device()
    used  = torch.cuda.memory_allocated(dev) / 1024**3
    total = torch.cuda.get_device_properties(dev).total_memory / 1024**3
    print(f"GPU        : {torch.cuda.get_device_name(dev)}")
    print(f"GPU 메모리 : {used:.1f} / {total:.1f} GB")
else:
    print("GPU        : 없음 (CPU 모드)")

logging.info("학습 시작")
best_te, model = train_gnn(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, args, data_config)

if best_te is not None:
    tn, fp, fn, tp = best_te['tn'], best_te['fp'], best_te['fn'], best_te['tp']
    total = tn + fp + fn + tp
    print("\nBest epoch test confusion matrix:")
    print(f"  TN: {tn:,} ({tn/total*100:.2f}%)  |  FP: {fp:,} ({fp/total*100:.2f}%)")
    print(f"  FN: {fn:,} ({fn/total*100:.2f}%)  |  TP: {tp:,} ({tp/total*100:.2f}%)")

## 7-1. XAI — TP/FP/FN/TN 그룹별 피처 중요도 분석 (선택)

학습 완료 후 실행하세요. 7번 셀이 먼저 실행되어 `model` 변수가 있어야 합니다.

- TP/FP/FN/TN 각 그룹에서 최대 500개씩 샘플링
- k-hop 서브그래프 고정 후 gradient saliency로 피처 중요도 계산
- `{unique_name}_feature_importance.csv` : 그룹별 평균·표준편차
- `{unique_name}_feature_importance_individual.csv` : 개별 샘플 결과

In [ ]:
import importlib.util, sys, torch
from pathlib import Path
from torch_geometric.loader import GraphSAINTRandomWalkSampler
from train_util import _to_pyg_data

# 파일 경로로 직접 임포트 — baseline/xai.py 충돌 완전 차단
_xai_path = Path(GRAPHSAINT_DIR) / 'xai.py'
_spec = importlib.util.spec_from_file_location('graphsaint_xai', _xai_path)
_xai_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_xai_mod)
run_xai = _xai_mod.run_xai

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# GraphData → PyG Data 변환 후 로더 생성 (get_loaders와 동일)
xai_te_loader = GraphSAINTRandomWalkSampler(
    _to_pyg_data(te_data),
    batch_size=args.saint_batch_size,
    walk_length=args.walk_length,
    num_steps=args.num_steps,
    sample_coverage=0,
    num_workers=0,
)

XAI_OUT = RUN_DIR / 'feature_importance'

summary_df, records_df = run_xai(
    te_loader = xai_te_loader,
    model     = model,
    te_data   = te_data,
    device    = device,
    args      = args,
    out_dir   = XAI_OUT,
    run_name  = args.unique_name,
    n_samples = 500,
)

print("
그룹별 평균 피처 중요도:")
mean_cols = [c for c in summary_df.columns if c.endswith('__mean')]
display(summary_df[mean_cols].round(4))

print(f"
저장 경로: {XAI_OUT}")
print(f"  - {args.unique_name}_feature_importance.csv")
print(f"  - {args.unique_name}_feature_importance_individual.csv")

## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}/runs